In [ ]:
import matplotlib.pyplot as plt
import matplotlib
import koreanize_matplotlib
import warnings

# 경고 무시 설정
warnings.filterwarnings("ignore")

# 한글 폰트 설정
matplotlib.rcParams['font.family'] = 'Malgun Gothic'  # 윈도우에서는 'Malgun Gothic'을 사용합니다.
matplotlib.rcParams['axes.unicode_minus'] = False  # 마이너스 기호 깨짐 방지

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon

# 1. 대전 구별 경계 파일 읽기 및 CRS 설정
jung_shp = gpd.read_file('C:/Users/kfq/Downloads/1ST_Final_Project/카셰어링/데이터/대전 데이터/지도/LX법정구역경계_시군구_대전광역시/대전광역시 구별 경계파일/중구.shp', encoding='utf-8-sig')
dong_shp = gpd.read_file('C:/Users/kfq/Downloads/1ST_Final_Project/카셰어링/데이터/대전 데이터/지도/LX법정구역경계_시군구_대전광역시/대전광역시 구별 경계파일/동구.shp', encoding='utf-8-sig')
seo_shp = gpd.read_file('C:/Users/kfq/Downloads/1ST_Final_Project/카셰어링/데이터/대전 데이터/지도/LX법정구역경계_시군구_대전광역시/대전광역시 구별 경계파일/서구.shp', encoding='utf-8-sig')
youseong_shp = gpd.read_file('C:/Users/kfq/Downloads/1ST_Final_Project/카셰어링/데이터/대전 데이터/지도/LX법정구역경계_시군구_대전광역시/대전광역시 구별 경계파일/유성구.shp', encoding='utf-8-sig')
daedeok_shp = gpd.read_file('C:/Users/kfq/Downloads/1ST_Final_Project/카셰어링/데이터/대전 데이터/지도/LX법정구역경계_시군구_대전광역시/대전광역시 구별 경계파일/대덕구.shp', encoding='utf-8-sig')

cty_shp = gpd.GeoDataFrame(pd.concat([jung_shp, dong_shp, seo_shp, youseong_shp, daedeok_shp], ignore_index=True))

# 좌표계 설정
cty_shp = cty_shp.to_crs(epsg=5179)

# 기준 지점 설정
orign = [cty_shp.unary_union.envelope.bounds[0], cty_shp.unary_union.envelope.bounds[1]]
dst = 1000  # 예시 거리 단위 (미터)
x_cnt = 10   # 예시 x 방향 셀 수
y_cnt = 10   # 예시 y 방향 셀 수

# grid cell 만들기
polygons = []
for xi in range(x_cnt):
    for yj in range(y_cnt):
        x1 = orign[0] + (dst * xi)
        x2 = x1 + dst
        y1 = orign[1] + (dst * yj)
        y2 = y1 + dst
        polygons.append(Polygon([(x1, y1), (x1, y2), (x2, y2), (x2, y1)]))

cty_grid = gpd.GeoDataFrame({'geometry': polygons}, crs=cty_shp.crs)
cty_grid = cty_grid[cty_grid.geometry.intersects(cty_shp.unary_union)]
cty_grid.reset_index(drop=True, inplace=True)

# CSV 파일을 읽어 DataFrame으로 변환 및 좌표계 변환
def read_csv_with_encoding(file_path, encodings=['utf-8', 'euc-kr', 'utf-8-sig', 'cp949']):
    if encodings is None:
        encodings = ['utf-8', 'euc-kr', 'utf-8-sig', 'cp949']  # 기본 인코딩 리스트
    
    for encoding in encodings:
        try:
            return pd.read_csv(file_path, encoding=encoding)
        except (UnicodeDecodeError, ValueError):
            continue
    raise ValueError(f"Failed to decode file with provided encodings: {encodings}")


def add_data_to_grid(csv_file_path, column_name, agg_func='count', is_value_col=False, encoding=None):
    data = read_csv_with_encoding(csv_file_path, encodings=[encoding] if encoding else None)
    gdf = gpd.GeoDataFrame(
        data,
        geometry=gpd.points_from_xy(data['경도'], data['위도']),
        crs="EPSG:4326"  # WGS84 좌표계
    )
    gdf = gdf.to_crs(cty_shp.crs)
    if is_value_col:
        cty_grid[column_name] = cty_grid.geometry.apply(lambda x: gdf[gdf.geometry.within(x)].sum()[column_name])
    else:
        cty_grid[column_name] = cty_grid.geometry.apply(lambda x: gdf[gdf.geometry.within(x)].shape[0])


# 차고지
add_data_to_grid("C:/Users/kfq/Downloads/1ST_Final_Project/카셰어링/데이터/대전 데이터/차고지/대전광역시 차고지 위치데이터(쏘카, 그린카).csv", '차고지')

# 시내버스정류장
add_data_to_grid("C:/Users/kfq/Downloads/1ST_Final_Project/카셰어링/데이터/대전 데이터/대중교통/대전광역시 버스정류장 위치정보.csv", '시내버스정류장')

# 기차역
add_data_to_grid("C:/Users/kfq/Downloads/1ST_Final_Project/카셰어링/데이터/대전 데이터/대중교통/대전광역시 기차역 데이터 (크롤링).csv", '기차역')

# 시외버스터미널
add_data_to_grid("C:/Users/kfq/Downloads/1ST_Final_Project/카셰어링/데이터/대전 데이터/대중교통/시외고속버스터미널 (버스타고).csv", '시외버스터미널')

# 도시철도
add_data_to_grid("C:/Users/kfq/Downloads/1ST_Final_Project/카셰어링/데이터/대전 데이터/대중교통/대전광역시 도시철도 위치정보.csv", '도시철도')

# 도시철도 시간대별 이동 패턴 분석
def add_time_slot_data(csv_file_path, time_slots):
    data = read_csv_with_encoding(csv_file_path, encodings=['euc-kr'])
    gdf = gpd.GeoDataFrame(
        data,
        geometry=gpd.points_from_xy(data['경도'], data['위도']),
        crs="EPSG:4326"  # WGS84 좌표계
    )
    gdf = gdf.to_crs(cty_shp.crs)
    for slot in time_slots:
        cty_grid[slot] = cty_grid.geometry.apply(lambda x: gdf[gdf.geometry.within(x)][slot].sum())

time_slots = [
    "03~04시", "04~05시", "05~06시", "06~07시", "07~08시", "08~09시", "09~10시", 
    "10~11시", "11~12시", "12~13시", "13~14시", "14~15시", "15~16시", "16~17시", 
    "17~18시", "18~19시", "19~20시", "20~21시", "21~22시", "22~23시", "23~00시", 
    "00~01시", "01~02시", "02~03시"
]
add_time_slot_data("C:/Users/kfq/Downloads/1ST_Final_Project/카셰어링/데이터/대전 데이터/대중교통/대전광역시 도시철도 시간대별 이동 패턴분석.csv", time_slots)

# 관광지
add_data_to_grid("C:/Users/kfq/Downloads/1ST_Final_Project/카셰어링/데이터/대전 데이터/관광지/대전광역시 관광지 데이터.csv", '관광지', encoding='euc-kr')

# 주차장
add_data_to_grid("C:/Users/kfq/Downloads/1ST_Final_Project/카셰어링/데이터/대전 데이터/주차장/대전광역시 유무료 주차장 정보 데이터.csv", '공영주차장', encoding='euc-kr')
add_data_to_grid("C:/Users/kfq/Downloads/1ST_Final_Project/카셰어링/데이터/대전 데이터/주차장/대전광역시 유무료 주차장 정보 데이터.csv", '주차구획수', is_value_col=True, encoding='euc-kr')

# 상권매출
add_data_to_grid("C:/Users/kfq/Downloads/1ST_Final_Project/카셰어링/데이터/대전 데이터/상권/대전광역시 상권매출.csv", '상권매출액', is_value_col=True)

# 행정동 GeoJSON 파일 읽기 및 병합
geojson_path = "C:/Users/kfq/Downloads/1ST_Final_Project/카셰어링/데이터/대전 데이터/지도/대전광역시 행정동.geojson"
gdf_adm = gpd.read_file(geojson_path)
gdf_adm = gdf_adm.to_crs(cty_shp.crs)

# 행정동 및 시군구 할당
def assign_adm_sgg(row, gdf):
    matching_gdf = gdf[gdf.geometry.intersects(row.geometry)]
    if not matching_gdf.empty:
        adm_nm = matching_gdf.iloc[0]['adm_nm']
        sggnm = matching_gdf.iloc[0]['sggnm']
        return pd.Series([adm_nm, sggnm], index=['adm_nm', 'sggnm'])
    return pd.Series([None, None], index=['adm_nm', 'sggnm'])

cty_grid[['adm_nm', 'sggnm']] = cty_grid.apply(lambda row: assign_adm_sgg(row, gdf_adm), axis=1)

# 최종 데이터프레임 출력
print(cty_grid)


In [ ]:
# 2. DataFrame을 GeoDataFrame으로 변환
gdf = gpd.GeoDataFrame(
    data,
    geometry=gpd.GeoSeries.from_wkt(data['geometry']),
    crs="EPSG:4326"  # WGS84 좌표계
)

# 좌표계 변환 (예를 들어, EPSG:5179로 변환)
gdf = gdf.to_crs(epsg=5179)


In [ ]:
gdf